# IITGN C2 Python Assignment
**Name:** PRITAM CHAVHAN 
**Student ID:** `26271029`  
**Programme:** PG Diploma in AI/ML and Agentic AI, Cohort-2



# Question 1: Clean Training Records Before Dataset Preparation

In [ ]:
from math import isfinite
from copy import deepcopy


def to_score(v):
    if isinstance(v, bool) or not isinstance(v, (int, float, str)):
        return None
    try:
        s = float(v)
    except (ValueError, OverflowError):
        return None
    return s if isfinite(s) and 0 <= s <= 100 else None


def clean_training_records(records):
    cleaned, rejected = [], []
    for i, r in enumerate(records):
        rec = r if isinstance(r, dict) else {}
        lid, score = rec.get("learner_id"), to_score(rec.get("score"))
        if isinstance(lid, str) and lid.strip() and score is not None:
            cleaned.append({"learner_id": lid.strip(), "score": score, "eligible": score >= 60})
        else:
            rejected.append(i)
    return {"cleaned": cleaned, "rejected_indices": rejected}

In [ ]:
records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": " ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]
print(clean_training_records(records))

In [ ]:
assert clean_training_records([]) == {"cleaned": [], "rejected_indices": []}

bad = ["text", {"learner_id": "A"}, {"score": 50}, {"learner_id": "B", "score": "nan"},
       {"learner_id": "C", "score": "inf"}, {"learner_id": "D", "score": 3 + 4j}, {"learner_id": 7, "score": 50}]
assert clean_training_records(bad)["rejected_indices"] == list(range(7))

edge = [{"learner_id": "a", "score": 0}, {"learner_id": "b", "score": 100}, {"learner_id": "c", "score": -1},
        {"learner_id": "d", "score": 100.1}, {"learner_id": "e", "score": " 59.9 "}, {"learner_id": "f", "score": "abc"}]
out = clean_training_records(edge)
assert [r["eligible"] for r in out["cleaned"]] == [False, True, False]
assert out["rejected_indices"] == [2, 3, 5]

dup = [{"learner_id": "X", "score": 70, "extra": 1}] * 2
assert len(clean_training_records(dup)["cleaned"]) == 2

snap = deepcopy(records)
clean_training_records(records)
assert records == snap
print("Q1 tests passed")

# Question 2: Prepare Mixed Text Payloads for an NLP Pipeline

In [ ]:
PUNCT = ".,!?;:"


def decode(p):
    if isinstance(p, str):
        return p
    if isinstance(p, (bytes, bytearray)):
        try:
            return bytes(p).decode("utf-8")
        except UnicodeDecodeError:
            return None
    return None


def prepare_messages(payloads, min_length=3):
    if not isinstance(min_length, int) or isinstance(min_length, bool) or min_length < 1:
        raise ValueError("min_length must be a positive integer")
    token_lists, rejected = [], []
    for i, p in enumerate(payloads):
        text = decode(p)
        if text is None:
            rejected.append(i)
            continue
        words = (w.strip(PUNCT) for w in text.lower().split())
        token_lists.append([w for w in words if len(w) >= min_length])
    vocab = sorted({t for toks in token_lists for t in toks})
    return {"token_lists": token_lists, "vocabulary": vocab, "rejected_indices": rejected}

In [ ]:
payloads = [" AI, helps TEAMS! ", b"Build reliable agents.", bytearray(b"AI helps."), None, b"\xff"]
print(prepare_messages(payloads, min_length=3))

In [ ]:
assert prepare_messages([]) == {"token_lists": [], "vocabulary": [], "rejected_indices": []}

out = prepare_messages(["   ", "", b"  "])
assert out["token_lists"] == [[], [], []] and out["vocabulary"] == []

assert prepare_messages(["agent-based, systems!!"])["token_lists"] == [["agent-based", "systems"]]
assert prepare_messages(["a bb ccc"], min_length=1)["token_lists"] == [["a", "bb", "ccc"]]
assert prepare_messages(["go go go"])["token_lists"] == [[]]
assert prepare_messages(["run run run"])["token_lists"] == [["run", "run", "run"]]

ba = bytearray(b"Hello World")
prepare_messages([ba])
assert ba == bytearray(b"Hello World")

for bad in (0, -1, True, "3", 2.5, None):
    try:
        prepare_messages(["abc"], min_length=bad)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass
print("Q2 tests passed")

# Question 3: Summarise Classification Prediction Results

In [ ]:
def summarise_predictions(results):
    by_actual, wrong, rejected = {}, [], []
    valid = correct = 0
    for i, r in enumerate(results):
        rec = r if isinstance(r, dict) else {}
        vals = [rec.get(k) for k in ("id", "actual", "predicted")]
        if not all(isinstance(v, str) and v.strip() for v in vals):
            rejected.append(i)
            continue
        rid, act, pred = (v.strip() for v in vals)
        act, pred = act.lower(), pred.lower()
        stats = by_actual.setdefault(act, {"total": 0, "correct": 0})
        stats["total"] += 1
        valid += 1
        if act == pred:
            stats["correct"] += 1
            correct += 1
        else:
            wrong.append(rid)
    return {
        "valid_count": valid,
        "correct_count": correct,
        "accuracy": round(correct / valid, 4) if valid else None,
        "by_actual": by_actual,
        "misclassified_ids": wrong,
        "rejected_indices": rejected,
    }

In [ ]:
results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]
print(summarise_predictions(results))

In [ ]:
assert summarise_predictions([]) == {
    "valid_count": 0, "correct_count": 0, "accuracy": None,
    "by_actual": {}, "misclassified_ids": [], "rejected_indices": [],
}

allbad = ["x", {"id": "1"}, {"id": " ", "actual": "a", "predicted": "a"}, {"id": "2", "actual": "a", "predicted": 5}]
out = summarise_predictions(allbad)
assert out["rejected_indices"] == [0, 1, 2, 3] and out["accuracy"] is None

rep = [{"id": "A", "actual": "x", "predicted": "y"}] * 3
out = summarise_predictions(rep)
assert out["valid_count"] == 3 and out["accuracy"] == 0.0
assert out["by_actual"] == {"x": {"total": 3, "correct": 0}}
assert out["misclassified_ids"] == ["A", "A", "A"]

third = [{"id": "1", "actual": "a", "predicted": "a", "note": "extra"},
         {"id": "2", "actual": "b", "predicted": "a"},
         {"id": "3", "actual": "b", "predicted": "b"}]
assert summarise_predictions(third)["accuracy"] == 0.6667

import copy
snap = copy.deepcopy(results)
summarise_predictions(results)
assert results == snap
print("Q3 tests passed")

# Question 4: Detect Dataset Duplicates and Overlap

In [ ]:
from collections import Counter


def is_id(x):
    return isinstance(x, str) and bool(x.strip())


def dupes(ids):
    return sorted(k for k, n in Counter(ids).items() if n > 1)


def audit_dataset_ids(train_ids, validation_ids):
    tr = [x.strip() for x in train_ids if is_id(x)]
    va = [x.strip() for x in validation_ids if is_id(x)]
    t, v = frozenset(tr), frozenset(va)
    return {
        "train_duplicates": dupes(tr),
        "validation_duplicates": dupes(va),
        "overlap": sorted(t & v),
        "train_only": sorted(t - v),
        "validation_only": sorted(v - t),
        "invalid_train_indices": [i for i, x in enumerate(train_ids) if not is_id(x)],
        "invalid_validation_indices": [i for i, x in enumerate(validation_ids) if not is_id(x)],
        "snapshots": {"train": t, "validation": v},
    }

In [ ]:
train_ids = ["R3", "R1", " R2 ", "R1", None]
validation_ids = ["R2", "R4", "R4", " "]
print(audit_dataset_ids(train_ids, validation_ids))

In [ ]:
out = audit_dataset_ids([], [])
assert out["overlap"] == [] and out["snapshots"] == {"train": frozenset(), "validation": frozenset()}

out = audit_dataset_ids(["a", "A", " a "], ["A"])
assert out["train_duplicates"] == ["a"]
assert out["overlap"] == ["A"] and out["train_only"] == ["a"] and out["validation_only"] == []

out = audit_dataset_ids([1, None, "", "  ", "ok"], [2.5, ("x",), "ok", "ok"])
assert out["invalid_train_indices"] == [0, 1, 2, 3]
assert out["invalid_validation_indices"] == [0, 1]
assert out["validation_duplicates"] == ["ok"] and out["overlap"] == ["ok"]

tr = ["x", "y"]
snap = audit_dataset_ids(tr, ["y"])["snapshots"]["train"]
tr.append("z")
tr[0] = "changed"
assert snap == frozenset({"x", "y"})
assert isinstance(snap, frozenset)
print("Q4 tests passed")

# Question 5: Update an Annotation Queue

In [ ]:
def update_annotation_queue(queue, incoming, completed, priority=None, batch_size=3):
    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 0:
        raise ValueError("batch_size must be a non-negative integer")
    seen, merged = set(), []
    for d in queue + incoming:
        if d not in seen:
            seen.add(d)
            merged.append(d)
    done = set(completed)
    merged = [d for d in merged if d not in done]
    if priority in merged:
        merged.remove(priority)
        merged.insert(0, priority)
    return {
        "queue": merged,
        "next_batch": merged[:batch_size],
        "last_item": merged[-1] if merged else None,
        "positions": [(i + 1, merged[i]) for i in range(len(merged))],
    }

In [ ]:
queue = ["D3", "D1", "D3", "D2"]
incoming = ["D4", "D2", "D5"]
completed = ["D1", "D9"]
print(update_annotation_queue(queue, incoming, completed, priority="D5", batch_size=3))
print(queue, incoming, completed)

In [ ]:
out = update_annotation_queue([], [], [])
assert out == {"queue": [], "next_batch": [], "last_item": None, "positions": []}

q, i, c = ["A", "B", "C"], ["D"], []
assert update_annotation_queue(q, i, c)["queue"] == ["A", "B", "C", "D"]
assert update_annotation_queue(q, i, c, priority="Z")["queue"] == ["A", "B", "C", "D"]
assert update_annotation_queue(q, i, c, priority="C")["queue"] == ["C", "A", "B", "D"]
assert update_annotation_queue(q, i, ["C"], priority="C")["queue"] == ["A", "B", "D"]

assert update_annotation_queue(q, i, c, batch_size=0)["next_batch"] == []
assert update_annotation_queue(q, i, c, batch_size=99)["next_batch"] == ["A", "B", "C", "D"]
assert update_annotation_queue(["a", "A"], [], [])["queue"] == ["a", "A"]

for bad in (-1, True, 1.5, "2", None):
    try:
        update_annotation_queue(q, i, c, batch_size=bad)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass

assert q == ["A", "B", "C"] and i == ["D"] and c == []
print("Q5 tests passed")

# Question 6: Build Independent Experiment Configurations

In [ ]:
from copy import deepcopy
from typing import Any


def build_experiment_config(
    base: dict[str, Any],
    *,
    run_name: str,
    parameter_updates: dict[str, Any] | None = None,
    **metadata: Any,
) -> dict[str, Any]:
    if not isinstance(run_name, str) or not run_name.strip():
        raise ValueError("run_name must be a non-empty string")
    if parameter_updates is not None and not isinstance(parameter_updates, dict):
        raise TypeError("parameter_updates must be a dict or None")
    updates = parameter_updates or {}
    unknown = updates.keys() - base["parameters"].keys()
    if unknown:
        raise KeyError(f"unknown parameters: {sorted(unknown)}")
    cfg = deepcopy(base)
    cfg["parameters"].update(updates)
    cfg["metadata"].update(metadata)
    cfg["run_name"] = run_name.strip()
    return cfg

In [ ]:
base = {
    "model": "ticket_classifier",
    "parameters": {"threshold": 0.5, "max_tokens": 128},
    "metadata": {"owner": "platform", "stage": "baseline"},
}
print(build_experiment_config(base, run_name="trial_02", parameter_updates={"threshold": 0.7}, owner="nlp_team", seed=42))
print(base)

In [ ]:
cfg = build_experiment_config(base, run_name="  r1  ")
assert cfg["run_name"] == "r1" and cfg["parameters"] == base["parameters"]
assert build_experiment_config(base, run_name="r", parameter_updates={})["parameters"] == base["parameters"]

cfg = build_experiment_config(base, run_name="r", parameter_updates={"max_tokens": 256}, stage="trial")
assert cfg["parameters"]["max_tokens"] == 256 and cfg["metadata"]["stage"] == "trial"
assert base["parameters"]["max_tokens"] == 128 and base["metadata"]["stage"] == "baseline"

for bad, err in (("", ValueError), ("   ", ValueError), (None, ValueError), (5, ValueError)):
    try:
        build_experiment_config(base, run_name=bad)
        raise AssertionError("expected error")
    except err:
        pass

for bad in ([("threshold", 1)], "x", 3):
    try:
        build_experiment_config(base, run_name="r", parameter_updates=bad)
        raise AssertionError("expected TypeError")
    except TypeError:
        pass

try:
    build_experiment_config(base, run_name="r", parameter_updates={"threshold": 1, "lr": 0.1})
    raise AssertionError("expected KeyError")
except KeyError:
    pass

a = build_experiment_config(base, run_name="a")
a["parameters"]["threshold"] = 0.99
a["metadata"]["owner"] = "someone_else"
assert base["parameters"]["threshold"] == 0.5 and base["metadata"]["owner"] == "platform"
base["parameters"]["max_tokens"] = 1
assert a["parameters"]["max_tokens"] == 128
base["parameters"]["max_tokens"] = 128
print("Q6 tests passed")

# Question 7: Aggregate Evaluation Scores Across Batches

In [ ]:
def is_score(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool) and 0 <= x <= 100


def aggregate_scores(*batches, top_n=3):
    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n < 0:
        raise ValueError("top_n must be a non-negative integer")
    if not all(isinstance(b, (list, tuple)) for b in batches):
        raise TypeError("every batch must be a list or tuple")
    found = [(b, i, float(x)) for b, batch in enumerate(batches) for i, x in enumerate(batch) if is_score(x)]
    rejected = [(b, i) for b, batch in enumerate(batches) for i, x in enumerate(batch) if not is_score(x)]
    mean = round(sum(s for _, _, s in found) / len(found), 2) if found else None
    top = sorted(found, key=lambda t: (-t[2], t[0], t[1]))[:top_n]
    return {"valid_count": len(found), "mean": mean, "top_scores": top, "rejected_positions": rejected}

In [ ]:
batch_a = [80, 90, None]
batch_b = (90, 70, True)
batch_c = []
print(aggregate_scores(batch_a, batch_b, batch_c, top_n=3))

In [ ]:
assert aggregate_scores() == {"valid_count": 0, "mean": None, "top_scores": [], "rejected_positions": []}
assert aggregate_scores([], ()) ["mean"] is None

out = aggregate_scores(["90", float("nan"), float("inf"), -1, 101, 50.5])
assert out["valid_count"] == 1 and out["mean"] == 50.5
assert out["rejected_positions"] == [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]

out = aggregate_scores([70, 70], [70], top_n=2)
assert out["top_scores"] == [(0, 0, 70.0), (0, 1, 70.0)]
assert aggregate_scores([1, 2, 3], top_n=0)["top_scores"] == []
assert aggregate_scores([0, 100])["mean"] == 50.0
assert aggregate_scores([1], [2], [4])["mean"] == 2.33

for bad in ("abc", {1: 2}, 5, None):
    try:
        aggregate_scores([1], bad)
        raise AssertionError("expected TypeError")
    except TypeError:
        pass

for bad in (-1, True, 2.0, "1"):
    try:
        aggregate_scores([1], top_n=bad)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass

data = [80, 90]
aggregate_scores(data)
assert data == [80, 90]
print("Q7 tests passed")

# Question 8: Organise Agent Tool Call Records

In [ ]:
from collections import defaultdict


def summarise_tool_calls(records):
    last = {r["call_id"]: r for r in records}
    groups = defaultdict(list)
    for r in last.values():
        groups[r["tool"]].append(r)
    by_tool = {
        t: {
            "calls": len(rs),
            "successes": sum(r["status"] == "ok" for r in rs),
            "failures": sum(r["status"] == "error" for r in rs),
            "total_duration_ms": sum(r["duration_ms"] for r in rs),
        }
        for t, rs in groups.items()
    }
    ranking = sorted(by_tool, key=lambda t: (-by_tool[t]["failures"], -by_tool[t]["calls"], t))
    failed = sorted(cid for cid, r in last.items() if r["status"] == "error")
    return {
        "unique_calls": len(last),
        "by_tool": by_tool,
        "failed_call_ids": failed,
        "tool_ranking": ranking,
    }

In [ ]:
records = [
    {"call_id": "C1", "tool": "search", "status": "error", "duration_ms": 120},
    {"call_id": "C2", "tool": "calculator", "status": "ok", "duration_ms": 10},
    {"call_id": "C1", "tool": "search", "status": "ok", "duration_ms": 80},
    {"call_id": "C3", "tool": "search", "status": "error", "duration_ms": 40},
    {"call_id": "C4", "tool": "retriever", "status": "error", "duration_ms": 50},
]
print(summarise_tool_calls(records))

In [ ]:
assert summarise_tool_calls([]) == {"unique_calls": 0, "by_tool": {}, "failed_call_ids": [], "tool_ranking": []}

moved = [
    {"call_id": "C1", "tool": "search", "status": "error", "duration_ms": 100},
    {"call_id": "C1", "tool": "retriever", "status": "ok", "duration_ms": 5},
]
out = summarise_tool_calls(moved)
assert list(out["by_tool"]) == ["retriever"] and out["by_tool"]["retriever"]["total_duration_ms"] == 5
assert out["failed_call_ids"] == []

same = [{"call_id": "C1", "tool": "t", "status": "ok", "duration_ms": 7}] * 3
assert summarise_tool_calls(same)["unique_calls"] == 1

ties = [
    {"call_id": "1", "tool": "b", "status": "error", "duration_ms": 1},
    {"call_id": "2", "tool": "a", "status": "error", "duration_ms": 1},
    {"call_id": "3", "tool": "c", "status": "error", "duration_ms": 1},
    {"call_id": "4", "tool": "c", "status": "ok", "duration_ms": 1},
    {"call_id": "5", "tool": "d", "status": "ok", "duration_ms": 1},
    {"call_id": "6", "tool": "e", "status": "ok", "duration_ms": 1},
    {"call_id": "7", "tool": "e", "status": "ok", "duration_ms": 1},
]
assert summarise_tool_calls(ties)["tool_ranking"] == ["c", "a", "b", "e", "d"]

import copy
snap = copy.deepcopy(records)
summarise_tool_calls(records)
assert records == snap
print("Q8 tests passed")

# Question 9: Validate Text Batches with a Reusable Decorator

In [ ]:
from functools import wraps


def validate_text_batch(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        texts = args[0] if args else kwargs.get("texts")
        if not isinstance(texts, list) or not all(isinstance(t, str) for t in texts):
            raise TypeError("texts must be a list of strings")
        if any(not t.strip() for t in texts):
            raise ValueError("texts must not contain blank strings")
        return func(*args, **kwargs)

    return wrapper


@validate_text_batch
def text_lengths(texts: list[str], *, strip: bool = True) -> list[int]:
    """Return the character count of each text."""
    return [len(t.strip() if strip else t) for t in texts]

In [ ]:
texts = ["  agent ", "ML", "data"]
print(text_lengths(texts))
print(text_lengths(texts=texts, strip=False))

In [ ]:
assert text_lengths([]) == []
assert text_lengths(["a b  c"]) == [6]
assert text_lengths(["  hi  "], strip=False) == [6]

for bad, err in (
    ("text", TypeError), (("a", "b"), TypeError), (None, TypeError), ([1, "a"], TypeError),
    ([""], ValueError), (["ok", "   "], ValueError), ([None, "  "], TypeError),
):
    try:
        text_lengths(bad)
        raise AssertionError("expected an error")
    except err:
        pass

try:
    text_lengths(texts=["ok", ""])
    raise AssertionError("expected ValueError")
except ValueError:
    pass

calls = []


@validate_text_batch
def probe(texts):
    calls.append(texts)
    return len(texts)


for bad in ([1], [" "], "abc"):
    try:
        probe(bad)
    except (TypeError, ValueError):
        pass
assert calls == []
assert probe(["x"]) == 1 and len(calls) == 1

assert text_lengths.__name__ == "text_lengths"
assert text_lengths.__doc__ == "Return the character count of each text."

data = [" a ", "b"]
text_lengths(data)
assert data == [" a ", "b"]
print("Q9 tests passed")

# Question 10: Track Successful and Failed Function Calls

In [ ]:
from functools import wraps
from math import isfinite


def track_calls(history):
    def deco(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            n = len(history) + 1

            def log(status, err=None):
                history.append({
                    "call_number": n,
                    "function": func.__name__,
                    "status": status,
                    "error_type": err,
                })

            try:
                result = func(*args, **kwargs)
            except Exception as e:
                log("error", type(e).__name__)
                raise
            log("success")
            return result

        return wrapper

    return deco


history = []


@track_calls(history)
def average_score(scores):
    """Return the mean of a non-empty list of scores."""
    if not scores:
        raise ValueError("scores must not be empty")
    return sum(scores) / len(scores)


@track_calls(history)
def format_run_name(prefix, *, number):
    """Return prefix and a two-digit number joined by an underscore."""
    return f"{prefix}_{number:02d}"

In [ ]:
print(average_score([60, 80]))
print(format_run_name("eval", number=3))
try:
    average_score([])
except ValueError as e:
    print("raised:", type(e).__name__)
for h in history:
    print(h)

In [ ]:
h1, h2 = [], []


@track_calls(h1)
def f(x, y=0, *rest, **opts):
    return None


@track_calls(h2)
def g():
    raise KeyError("k")


assert f(1, 2, 3, z=4) is None
assert h1 == [{"call_number": 1, "function": "f", "status": "success", "error_type": None}]

try:
    g()
except KeyError:
    pass
assert h2 == [{"call_number": 1, "function": "g", "status": "error", "error_type": "KeyError"}]
assert len(h1) == 1

shared = []


@track_calls(shared)
def a():
    return "a"


@track_calls(shared)
def b():
    raise RuntimeError


a()
try:
    b()
except RuntimeError:
    pass
a()
assert [r["call_number"] for r in shared] == [1, 2, 3]
assert [r["status"] for r in shared] == ["success", "error", "success"]

assert average_score.__name__ == "average_score"
assert format_run_name.__doc__ == "Return prefix and a two-digit number joined by an underscore."
assert format_run_name("run", number=12) == "run_12"
assert average_score([1.5, 2.5]) == 2.0

class Boom(Exception):
    pass


@track_calls(h2)
def h():
    raise Boom


try:
    h()
except Boom as e:
    assert type(e) is Boom
assert h2[-1]["error_type"] == "Boom" and h2[-1]["call_number"] == 2
print("Q10 tests passed")